# &nbsp;&nbsp;&nbsp;&nbsp;<span style="color: #1387abff;">Side Chapter1: </span>一个强化学习项目

#### 哇，不知道的还以为笔者已经不在人世或者跑到完全没有网络的深山老林里隐居了呢！如果笔者真的能在荒无人烟的地方训练出Gemini3pro级别的智能体，笔者说不定真的不回来写教程了。但是实际上笔者只是花了3个月去实践一个强化学习项目。今天是2026年的第二天，由于实在无法忍受对于贝勒爷积分（或者勒布朗积分 勒贝格积分）枯燥繁重的复习，笔者决定回来讲讲这个强化学习项目做了什么。

#### 这一章是番外篇，你可以理解为新世纪福音战士的OVA，或者死亡笔记的洛杉矶B.B.杀人案。理论上跳过对这章的阅读完全不会影响你接下来的人工智能学习之路，但是笔者非常建议你先把这章读完。

#### 简而言之，我们制作了一款游戏，名为CALC WARS。你可以在笔者的GitHub主页看到这个项目的代码仓库，其中还包含了一份技术报告，里面详细记录了这个项目所有的技术细节。游戏是传统的双人对战回合制卡牌游戏，玩家需要通过组合手中卡片（数字或者加减乘除括号）来计算特定结果，而特殊的计算数字会给予玩家特殊的效果。尽管我们为游戏平衡性设计了很久，但是游戏本体并不是重点，你只需要知道这个游戏策略深度不低且是回合制卡牌游戏即可。项目的重心在于我们为这款游戏制作了一个AI对手，可以作为玩家强大的陪练，对新手单局胜率超越了70%。更重要的是，这个模型仅仅4.5M参数量，并且是不配备MCTS搜索的纯直觉网络。这意味着这是一个绝佳的强化学习教学案例。

#### 番外篇经常做的是就是添加主线没提到的设定，笔者现在把强化学习的概念引入这个世界。简单来说，传统的机器学习就是把训练数据集全部写好了，强化学习则是模型从与环境的交互中学习训练数据集。一般来说，人类学习技能的过程也是一种强化学习。

#### 在强化学习的框架中，所有的交互都可以简化为五个基本组件的循环：
##### 智能体 (Agent)： 学习者，也就是“大脑”。它决定在什么情况下采取什么行动。
##### 环境 (Environment)： 智能体所处的世界。它接收智能体的动作，并给出反馈。
##### 状态 ($S$ - State)： 智能体在某一时刻感知的环境情况。
##### 动作 ($A$ - Action)： 智能体做出的选择。
##### 奖励 ($R$ - Reward)： 环境对动作的实时打分。这是“强化”的关键。

#### <span style="color: #c50870a4;">那么，这个智能体的目标是什么？
#### 智能体的终极目标不是获得当下的最高分，而是获得长期累积奖励的最大化。数学上，我们通常用折扣奖励和（Discounted Return）来表示：
$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$
#### 其中 $\gamma$ 是折扣因子，代表了智能体对“眼前利益”和“长远利益”的权衡。

#### 我想你一定立刻懂了：既然我们需要总分最大，直接将总分的负数作为损失函数吧。在一局游戏结束之后，我们依靠PyTorch强大的自动微分进行梯度更新。这样就转化成传统的机器学习问题了！

#### <span style = "color: #930ecc9b;">等等，似乎哪里不对？
#### 不要太相信你的直觉，上面这个思路接近正确：在合理建模损失函数之后，我们确实可以依赖自动微分来梯度更新，转化成我们已知情况。然而笔者的潜台词是，直接将总分负数作为损失函数是不合理的，因为PyTorch不知道这个函数长什么样，自动微分根本无从下手！！
#### 何意味？拿监督学习来说，就比如最基本的MNIST手写数字识别，模型输出概率分布之后，我们其实手动为模型输出与标准答案做了交叉熵，得到了损失函数。换句话说，我们告诉了PyTorch这个函数的样子，所以梯度得以传播。然而，强化学习中的"总分"的评判规则是完全不知道的，甚至知道了也是不可导的。因此这样直接的建模是不可取的。

#### 如果哪位读者读到这段很不服气，因而被激励发明了黑箱式的自动微分，完全颠覆强化学习领域几十年发展，届时请不要忘记笔者啊！！那么愚蠢又智慧的先古学者如何建模这个损失函数呢？他们如何改造这个"不可微分"的函数？笔者将为你介绍策略梯度定理（Policy Gradient Theorem）。

#### 为了解决这个难题，数学家们利用了一个极其简单的微积分恒等式：$$\nabla_\theta \pi_\theta(x) = \pi_\theta(x) \frac{\nabla_\theta \pi_\theta(x)}{\pi_\theta(x)} = \pi_\theta(x) \nabla_\theta \log \pi_\theta(x)$$这个技巧的神奇之处在于：它把概率密度的梯度转变成了概率密度本身乘以对数概率的梯度。

#### 这个公式在说什么？？别着急，我们先明确我们需要优化什么：我们不是要优化某一局的$G_t$，而是要优化一个总的期望奖励$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} [R(\tau)]$$或者让我们把这个式子展开：$$J(\theta) = \sum_{\tau} P(\tau|\theta) R(\tau)$$其中$P(\tau|\theta)$是在$\theta$权重下一局内完成轨迹$\tau$的可能性，而$R(\tau)$则是在轨迹$\tau$下得到的一局奖励。这看起来很好理解，比如模型玩一局游戏50%大获全胜，得1分；30%平手，得0分；20%大败而归，得-1分。那么期望奖励就是1 * 0.5+0 * 0.3+(-1) * 0.2=0.3分。

#### 所以，按照我们梯度下降的思想，我们最好把$\nabla_\theta J(\theta)$想办法写出来。请看下面这个神奇的变换：$$\begin{aligned} \nabla_\theta J(\theta) &= \nabla_\theta \int \pi_\theta(\tau) R(\tau) d\tau & \text{(将期望写成积分形式)} \\ &= \int \nabla_\theta \pi_\theta(\tau) R(\tau) d\tau & \text{(梯度符号移入积分)} \\ &= \int \pi_\theta(\tau) \nabla_\theta \log \pi_\theta(\tau) R(\tau) d\tau & \text{(代入对数导数技巧)} \\ &= \mathbb{E}_{\tau \sim \pi_\theta} [\nabla_\theta \log \pi_\theta(\tau) R(\tau)] & \text{(重新写回期望形式)} \end{aligned}$$

#### 在这里的符号中，$\nabla_\theta \pi_\theta(\tau)$和$P(\tau|\theta)$其实是同一个意思：在参数为 $\theta$ 的策略下，出现某条特定路径（轨迹） $\tau$ 的概率，也就是概率密度函数。更多的，$$\pi_\theta(\tau) = P(s_0) \prod_{t=0}^{T} \pi_\theta(a_t|s_t) P(s_{t+1}|s_t, a_t)$$ $$\log \pi_\theta(\tau) = \log P(s_0) + \sum (\log \pi_\theta(a_t|s_t) + \log P(s_{t+1}|s_t, a_t))$$ $$\nabla_\theta \log \pi_\theta(\tau) = \sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t)$$所以我们得到$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot R(\tau) \right]$$

#### 回到我们先前的梯度上升。我们需要让期望奖励最大，因此权重更新公式是：$$\theta_{new} = \theta_{old} + \alpha \cdot \nabla_\theta J(\theta)$$

#### 然而，这里我们需要处理一个工程问题，那就是实际训练中我们是无法采样所有轨迹的，因此也无从计算精确的$\nabla_\theta J(\theta)$。现实的做法是，我们让模型玩一局之后，近似这个期望梯度：$$\theta \leftarrow \theta + \alpha \cdot \left( \nabla_\theta \log \pi_\theta(\tau) \cdot R(\tau) \right)$$拆开来就是$$\theta \leftarrow \theta + \alpha \cdot \left( \sum_{t=0}^{T} \nabla_\theta \log \pi_\theta(a_t|s_t) \right) \cdot R(\tau)$$

#### <span style="color: #1488dbd3;">至此，你已经学习了现代强化学习的梯度更新雏形。
#### 然而这还远远不够。以上算法有个重大问题就是梯度极度不稳定和极度延迟奖励。如果模型随便玩一玩但是运气非常好，算法也会忠实地大幅提升这些"随便玩一玩动作"出现的概率，导致模型训练非常不稳定。更多的，我们把每一步动作都乘以一个最终奖励，其实反而没有告诉模型"哪一步是神来之笔"。因此为了解决这些痛点，笔者将为你介绍更成熟的<span style="color: #db1442d3;">REINFORCE算法。

#### REINFORCE最核心的修改是：因果律（Causality）。模型在第 50 步出的牌，只能影响第 50 步之后的奖励，不可能影响第 10 步已经拿到的奖励。说严谨些，由于第 $t$ 步之前的奖励已经发生了，它们和你在第 $t$ 步如何调整参数 $\theta$ 没有任何逻辑上的因果关系，换句话说$\nabla_\theta \log \pi_\theta(a_t|s_t)$ 与 $t$ 时刻之前的奖励的乘积，其期望值为 0。因此，REINFORCE 不再使用整局的总奖励 $R(\tau)$，而是使用 $t$ 时刻之后的累积回报 $G_t$：$$G_t = \sum_{k=t}^{T} \gamma^{k-t} r_k$$

#### 最终，REINFORCE算法的更新方式是：$$\theta \leftarrow \theta + \alpha \sum_{t=0}^{T} \left( \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot G_t \right)$$

#### 最原教旨主义的REINFORCE是每玩一局更新一次的，因为需要考察当前权重下梯度。然而现实工程中，我们会为了梯度稳定做改良，也就是采样多局更新一次，这样梯度会稳定得多，同时也更接近我们理想的期望奖励优化：$$\nabla J(\theta) \approx \frac{1}{M} \sum_{i=1}^M \left( \sum_{t} \nabla \log \pi \cdot G_{i,t} \right)$$

#### 然而，无论是一局更新还是十局更新，REINFORCE 有一个必须遵守原则：它是 On-policy（同策略）算法。这就意味着：你必须用当前的策略 $\pi_\theta$ 去玩，计算出梯度 $\nabla_\theta$，更新完 $\theta$ 之后，刚才采样的那些数据必须立刻扔掉。如果你对同一批数据更新了两次 $\theta$，那么在第二次更新时，你的 $\theta$ 已经变了，而数据还是旧的 $\theta_{old}$ 产生的。这就不再满足 $\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} [\dots]$ 的期望定义了。

#### 你可以把 REINFORCE 的运作流程想象成这样：打完第 1 局 $\to$ 计算 $G_t$ $\to$ （存入 Batch）打完第 2 局 $\to$ 计算 $G_t$ $\to$ （存入 Batch）...凑够 $M$ 局 $\to$ 求平均梯度 $\to$ 更新一次权重 $\theta$ $\to$ 清空 Batch，刚才的数据全部丢弃了。

#### 为那些被丢弃的数据可惜吧！我们辛辛苦苦采样的数据，更新一次就用完了，仅仅是因为强烈的On-Policy束缚。我们希望对同一批数据多更新几次，这样会大大加速收敛进程。我们为此引入重要性采样（Importance Sampling）。同时，REINFORCE算法实际上方差会很大，因为实际上累计回报并不能精确估计"这一步有多么厉害"。我们为此引入基准值（Baseline）。

#### 重要性采样是一个通用的概率论技巧：如果你想计算函数 $f(x)$ 在分布 $p$ 下的期望，但你只有分布 $q$ 的数据，你可以通过一个修正系数来实现：$$\mathbb{E}_{x \sim p}[f(x)] = \int p(x) f(x) dx = \int q(x) \frac{p(x)}{q(x)} f(x) dx = \mathbb{E}_{x \sim q}\left[ \frac{p(x)}{q(x)} f(x) \right]$$这里 $\frac{p(x)}{q(x)}$ 被称为 重要性权重（Importance Weight）。

#### 应用到策略梯度中，即便我们手里的是旧策略 $\pi_{\theta_{old}}$ 跑出来的数据，我们也可以更新新策略 $\pi_\theta$：$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_{\theta_{old}}} \left[ \frac{\pi_\theta(\tau)}{\pi_{\theta_{old}}(\tau)} \nabla_\theta \log \pi_\theta(\tau) R(\tau) \right]$$

#### 有了这个公式，Batch数据终于可以反复榨取了！用 $\theta_{old}$ 跑 100 局存起来，对这 100 局数据进行第 1 次更新得到 $\theta_1$，第 2 次更新得到 $\theta_2 \dots$。次更新时，只需计算 $\frac{\pi_{new}}{\pi_{old}}$ 这个比值，就能在数学上保证更新方向依然是正确的。

#### 工程上的细节是，我们需要把上式展开到每一步：$$\pi_\theta(\tau) = P(s_0) \prod_{k=0}^T \pi_\theta(a_k|s_k) P(s_{k+1}|s_k, a_k)$$ $$\frac{\pi_\theta(\tau)}{\pi_{\theta_{old}}(\tau)} = \prod_{k=0}^T \frac{\pi_\theta(a_k|s_k)}{\pi_{\theta_{old}}(a_k|s_k)}$$ $$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_{\theta_{old}}} \left[ \left( \prod_{k=0}^T \frac{\pi_\theta(a_k|s_k)}{\pi_{\theta_{old}}(a_k|s_k)} \right) \left( \sum_{t=0}^T \nabla_\theta \log \pi_\theta(a_t|s_t) \right) R(\tau) \right]$$

#### 虽然重要性采样解决了效率问题，但它甚至可能放大方差（如果两个分布差得太远）。所以在引入重要性采样之前或同时，人们通常会给REINFORCE做一个更简单的改良：将 $G_t$ 替换为 $(G_t - b(s))$。其中 $b(s)$ 就是 基准值（Baseline），通常由一个预测当前局面平均分的神经网络（Critic）担任。